<a href="https://colab.research.google.com/github/JeevanS-0721/Employee-Wellness-Management-/blob/main/Authentication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |


neondb_owner

In [1]:
!pip install -q streamlit==1.38.0 psycopg2-binary==2.9.9 PyJWT==2.9.0 bcrypt==4.2.0 \
    python-dotenv==1.0.1 email-validator==2.2.0 pyngrok==7.2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.3.0 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.
google-adk 2.3.0 requires watchdog<7,>=6, but you have watchdog 4.0.2 which is incompatible.


In [2]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [3]:
%%writefile db.py
"""
db.py
PostgreSQL connection handling + schema initialization.
"""

import os
import psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "dbname": os.getenv("DB_NAME", "auth_app"),
    "user": os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", ""),
    "sslmode": "require",
}


@contextmanager
def get_connection():
    """Yield a PostgreSQL connection, always closed on exit."""
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        yield conn
    finally:
        conn.close()


@contextmanager
def get_cursor(commit: bool = False):
    """Yield a RealDictCursor. Commits if commit=True and no exception occurred."""
    with get_connection() as conn:
        cur = conn.cursor(cursor_factory=RealDictCursor)
        try:
            yield cur
            if commit:
                conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            cur.close()


def init_db():
    """Create tables if they don't exist. Safe to call on every app startup."""
    with get_cursor(commit=True) as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id SERIAL PRIMARY KEY,
                username VARCHAR(50) UNIQUE NOT NULL,
                email VARCHAR(255) UNIQUE NOT NULL,
                password_hash VARCHAR(255) NOT NULL,
                is_verified BOOLEAN NOT NULL DEFAULT FALSE,
                created_at TIMESTAMP NOT NULL DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS otp_codes (
                id SERIAL PRIMARY KEY,
                email VARCHAR(255) NOT NULL,
                code VARCHAR(6) NOT NULL,
                purpose VARCHAR(20) NOT NULL,
                expires_at TIMESTAMP NOT NULL,
                used BOOLEAN NOT NULL DEFAULT FALSE,
                created_at TIMESTAMP NOT NULL DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_otp_email_purpose
            ON otp_codes (email, purpose);
        """)
    print("Database initialized (tables ensured).")


if __name__ == "__main__":
    init_db()

Writing db.py


In [4]:
%%writefile auth.py
"""
auth.py
Password hashing, JWT issuing/verification, and user data-access functions.
"""

import os
import jwt
import bcrypt
import random
import string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

from db import get_cursor

load_dotenv()

JWT_SECRET = os.getenv("JWT_SECRET")
JWT_ALGORITHM = os.getenv("JWT_ALGORITHM", "HS256")
JWT_EXPIRY_MINUTES = int(os.getenv("JWT_EXPIRY_MINUTES", "60"))
OTP_EXPIRY_MINUTES = int(os.getenv("OTP_EXPIRY_MINUTES", "10"))


def hash_password(plain_password: str) -> str:
    return bcrypt.hashpw(plain_password.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")


def verify_password(plain_password: str, password_hash: str) -> bool:
    return bcrypt.checkpw(plain_password.encode("utf-8"), password_hash.encode("utf-8"))


def create_jwt(user_id: int, username: str, email: str) -> str:
    payload = {
        "user_id": user_id,
        "username": username,
        "email": email,
        "exp": datetime.now(timezone.utc) + timedelta(minutes=JWT_EXPIRY_MINUTES),
        "iat": datetime.now(timezone.utc),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)


def decode_jwt(token: str):
    """Returns the payload dict if valid, else None."""
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None


def get_user_by_email(email: str):
    with get_cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email = %s;", (email,))
        return cur.fetchone()


def get_user_by_username(username: str):
    with get_cursor() as cur:
        cur.execute("SELECT * FROM users WHERE username = %s;", (username,))
        return cur.fetchone()


def create_user(username: str, email: str, password: str):
    password_hash = hash_password(password)
    with get_cursor(commit=True) as cur:
        cur.execute(
            """
            INSERT INTO users (username, email, password_hash, is_verified)
            VALUES (%s, %s, %s, FALSE)
            RETURNING id, username, email;
            """,
            (username, email, password_hash),
        )
        return cur.fetchone()


def mark_user_verified(email: str):
    with get_cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified = TRUE WHERE email = %s;", (email,))


def update_user_password(email: str, new_password: str):
    password_hash = hash_password(new_password)
    with get_cursor(commit=True) as cur:
        cur.execute(
            "UPDATE users SET password_hash = %s WHERE email = %s;",
            (password_hash, email),
        )


def generate_otp() -> str:
    return "".join(random.choices(string.digits, k=6))


def store_otp(email: str, code: str, purpose: str):
    """purpose: 'signup' or 'password_reset'"""
    expires_at = datetime.now(timezone.utc) + timedelta(minutes=OTP_EXPIRY_MINUTES)
    with get_cursor(commit=True) as cur:
        cur.execute(
            "UPDATE otp_codes SET used = TRUE WHERE email = %s AND purpose = %s AND used = FALSE;",
            (email, purpose),
        )
        cur.execute(
            """
            INSERT INTO otp_codes (email, code, purpose, expires_at)
            VALUES (%s, %s, %s, %s);
            """,
            (email, code, purpose, expires_at),
        )


def verify_otp(email: str, code: str, purpose: str) -> bool:
    with get_cursor(commit=True) as cur:
        cur.execute(
            """
            SELECT * FROM otp_codes
            WHERE email = %s AND purpose = %s AND used = FALSE
            ORDER BY created_at DESC LIMIT 1;
            """,
            (email, purpose),
        )
        row = cur.fetchone()
        if not row:
            return False

        expires_at = row["expires_at"]
        now = datetime.now(expires_at.tzinfo) if expires_at.tzinfo else datetime.now()

        if now > expires_at:
            return False
        if row["code"] != code:
            return False

        cur.execute("UPDATE otp_codes SET used = TRUE WHERE id = %s;", (row["id"],))
        return True

Writing auth.py


In [5]:
%%writefile email_utils.py
"""
email_utils.py
Sends OTP emails using Gmail SMTP (smtp.gmail.com, port 587, STARTTLS).
"""

import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv

load_dotenv()

SMTP_HOST = os.getenv("SMTP_HOST", "smtp.gmail.com")
SMTP_PORT = int(os.getenv("SMTP_PORT", "587"))
SMTP_EMAIL = os.getenv("SMTP_EMAIL")
SMTP_APP_PASSWORD = os.getenv("SMTP_APP_PASSWORD")


def send_email(to_email: str, subject: str, body: str):
    if not SMTP_EMAIL or not SMTP_APP_PASSWORD:
        return False, "SMTP credentials are not configured in .env"

    msg = MIMEMultipart()
    msg["From"] = SMTP_EMAIL
    msg["To"] = to_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=15) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(SMTP_EMAIL, SMTP_APP_PASSWORD)
            server.sendmail(SMTP_EMAIL, to_email, msg.as_string())
        return True, "Email sent."
    except smtplib.SMTPAuthenticationError:
        return False, "SMTP auth failed. Check SMTP_EMAIL / SMTP_APP_PASSWORD (use a Gmail App Password)."
    except Exception as e:
        return False, f"Failed to send email: {e}"


def send_otp_email(to_email: str, otp_code: str, purpose: str):
    if purpose == "signup":
        subject = "Your Verification Code"
        body = (
            f"Welcome!\n\nYour verification code is: {otp_code}\n\n"
            f"This code expires in a few minutes. If you did not request this, "
            f"you can safely ignore this email."
        )
    else:
        subject = "Your Password Reset Code"
        body = (
            f"We received a request to reset your password.\n\n"
            f"Your reset code is: {otp_code}\n\n"
            f"This code expires in a few minutes. If you did not request this, "
            f"you can safely ignore this email."
        )
    return send_email(to_email, subject, body)

Writing email_utils.py


In [9]:
%%writefile app.py
import streamlit as st
import re
import random
import os
import datetime
from dotenv import load_dotenv

# Import database functions safely from your companion cells
from db import get_cursor
from auth import (
    hash_password, verify_password, create_jwt, decode_jwt,
    get_user_by_email, get_user_by_username, create_user,
    mark_user_verified, update_user_password, generate_otp,
    store_otp, verify_otp
)
# Import the underlying core mailing function directly
import email_utils

load_dotenv()

# 1. Page Configuration
st.set_page_config(
    page_title="MoodMentor",
    page_icon="😊",
    layout="centered"
)

# 2. Enforce Light Shade of Green Theme with High Contrast Black Text
st.markdown("""
    <style>
    .stApp, [data-testid="stAppViewContainer"] {
        background-color: #F0FDF4 !important;
        color: #0F172A !important;
    }
    p, span, li, label, div {
        color: #0F172A !important;
    }
    h1, h2, h3, [data-testid="stMarkdownContainer"] h1, [data-testid="stMarkdownContainer"] h2, [data-testid="stMarkdownContainer"] h3 {
        color: #166534 !important;
        font-weight: 700 !important;
    }
    [data-testid="stWidgetLabel"] p, label p, .stSlider label {
        color: #0F172A !important;
        font-weight: 600 !important;
    }
    [data-testid="stSidebar"] {
        background-color: #FFFFFF !important;
        border-right: 1px solid #BBF7D0 !important;
    }
    input, textarea, [data-baseweb="input"], [data-baseweb="select"] {
        background-color: #FFFFFF !important;
        color: #0F172A !important;
        border: 1px solid #86EFAC !important;
        border-radius: 6px !important;
    }
    input {
        -webkit-text-fill-color: #0F172A !important;
    }
    [data-testid="stSidebar"] div[data-baseweb="select"] > div {
        background-color: #FFFFFF !important;
        color: #0F172A !important;
        border: 1px solid #86EFAC !important;
    }
    [data-baseweb="popover"], [data-baseweb="menu"], ul[role="listbox"], li[role="option"] {
        background-color: #FFFFFF !important;
        color: #0F172A !important;
    }
    [data-testid="stSidebar"] div[data-baseweb="select"] span, li[role="option"] span {
        color: #0F172A !important;
        -webkit-text-fill-color: #0F172A !important;
    }
    div.stButton > button {
        background-color: #2563EB !important;
        color: #FFFFFF !important;
        border: none !important;
        border-radius: 6px !important;
        font-weight: 600 !important;
        padding: 0.5rem 1rem !important;
    }
    div.stButton > button:hover {
        background-color: #1D4ED8 !important;
        color: #FFFFFF !important;
    }
    </style>
""", unsafe_allow_html=True)

# Session State Persistence
if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "user_email" not in st.session_state:
    st.session_state.user_email = ""
if "user_username" not in st.session_state:
    st.session_state.user_username = ""
if "signup_email_target" not in st.session_state:
    st.session_state.signup_email_target = ""
if "forgot_target_email" not in st.session_state:
    st.session_state.forgot_target_email = ""

# Navigation Manager
if st.session_state.authenticated:
    menu_options = ["📊 Dashboard", "🚪 Logout"]
else:
    menu_options = ["🏠 Home", "📝 Sign Up", "🔐 Login", "🔄 Forgot Password"]

choice = st.sidebar.selectbox("Navigation Menu", menu_options)

# ---------------- 1. HOME SCREEN ----------------
if choice == "🏠 Home":
    st.title("😊 MoodMentor")
    st.write("Welcome to MoodMentor. Monitor, analyze, and map employee wellness trends cleanly.")
    st.info("💡 Use the sidebar menu to register or access your secure portal.")

# ---------------- 2. SIGN UP WITH VALIDATION & OTP ----------------
elif choice == "📝 Sign Up":
    st.title("📝 Create Account")
    reg_username = st.text_input("Username")
    reg_email = st.text_input("Email Address")
    reg_pass = st.text_input("Password", type="password")
    reg_confirm = st.text_input("Confirm Password", type="password")

    email_regex = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"

    if st.button("Send Verification OTP"):
        if not reg_username or not reg_email or not reg_pass:
            st.error("All registration input values must be filled out completely.")
        elif reg_pass != reg_confirm:
            st.error("Passwords do not match.")
        elif not re.match(email_regex, reg_email):
            st.error("Invalid email address format.")
        else:
            if get_user_by_email(reg_email):
                st.error("This email is already registered.")
            elif get_user_by_username(reg_username):
                st.error("This username is already taken.")
            else:
                # Force runtime injection of the system variables directly to the imported module variables
                email_utils.SMTP_EMAIL = os.getenv("SMTP_EMAIL")
                email_utils.SMTP_APP_PASSWORD = os.getenv("SMTP_APP_PASSWORD")

                code = generate_otp()
                store_otp(reg_email, code, "signup")
                success, msg = email_utils.send_otp_email(reg_email, code, "signup")
                if success:
                    st.session_state.signup_email_target = reg_email
                    st.success(f"Verification code sent successfully to {reg_email}!")
                else:
                    st.error(f"SMTP Transmission Failed: {msg}")

    input_signup_otp = st.text_input("Enter 6-Digit Email OTP")
    if st.button("Verify OTP & Create Profile"):
        if not st.session_state.signup_email_target:
            st.error("Please request a verification token code before verifying.")
        elif not reg_username or not reg_email:
            st.error("Form fields missing data inputs.")
        else:
            if verify_otp(st.session_state.signup_email_target, input_signup_otp, "signup"):
                new_user = create_user(reg_username, reg_email, reg_pass)
                mark_user_verified(reg_email)
                st.session_state.signup_email_target = ""
                st.success("Account created and verified successfully! Switch to the Login menu.")
            else:
                st.error("Incorrect or expired verification code.")

# ---------------- 3. LOGIN SCREEN ----------------
elif choice == "🔐 Login":
    st.title("🔐 User Authentication")
    login_email = st.text_input("Email Address")
    login_password = st.text_input("Password", type="password")

    if st.button("Authenticate"):
        user_record = get_user_by_email(login_email)

        if user_record and verify_password(login_password, user_record["password_hash"]):
            token = create_jwt(user_record["id"], user_record["username"], user_record["email"])
            st.session_state.authenticated = True
            st.session_state.user_email = user_record["email"]
            st.session_state.user_username = user_record["username"]
            st.success("Access Granted! Cryptographic user session authenticated.")
            st.rerun()
        else:
            st.error("Access Denied: Invalid email address or account password.")

# ---------------- 4. FORGOT PASSWORD SCREEN ----------------
elif choice == "🔄 Forgot Password":
    st.title("🔄 Reset Account Credentials")

    target_email = st.text_input("Enter Registered Email Address")
    if st.button("Send Reset Token Code"):
        user_record = get_user_by_email(target_email)
        if user_record:
            # Force runtime injection of the system variables directly to the imported module variables
            email_utils.SMTP_EMAIL = os.getenv("SMTP_EMAIL")
            email_utils.SMTP_APP_PASSWORD = os.getenv("SMTP_APP_PASSWORD")

            code = generate_otp()
            store_otp(target_email, code, "password_reset")
            success, msg = email_utils.send_otp_email(target_email, code, "password_reset")
            if success:
                st.session_state.forgot_target_email = target_email
                st.success(f"Security recovery code sent safely to {target_email}.")
            else:
                st.error(f"SMTP Gateway transmission dropped: {msg}")
        else:
            st.error("This email address is not registered in our records.")

    input_forgot_otp = st.text_input("Enter 6-Digit Recovery OTP")
    new_password = st.text_input("New Password String", type="password")
    confirm_new_password = st.text_input("Confirm New Password String", type="password")

    if st.button("Update Database Encryption Profile"):
        if not st.session_state.forgot_target_email:
            st.error("Please initiate a recovery pass request first.")
        elif new_password != confirm_new_password:
            st.error("Passwords do not match.")
        else:
            if verify_otp(st.session_state.forgot_target_email, input_forgot_otp, "password_reset"):
                update_user_password(st.session_state.forgot_target_email, new_password)
                st.session_state.forgot_target_email = ""
                st.success("Password update complete! You can now navigate back to Login.")
            else:
                st.error("Token verification match failed or expired.")

# ---------------- 5. AUTHENTICATED DASHBOARD ----------------
elif choice == "📊 Dashboard":
    st.title("📊 Enterprise Well-Being Analytics Dashboard")
    st.caption(f"Secure Cryptographic Session Active: {st.session_state.user_username} ({st.session_state.user_email})")

    col1, col2 = st.columns(2)
    with col1:
        st.metric(label="Current Team State Balance", value="😊 Balanced", delta="Stable")
    with col2:
        st.metric(label="System Core Polarity", value="Positive Baseline", delta="91% Accuracy Target")

# ---------------- 6. LOGOUT CLEANUP ----------------
elif choice == "🚪 Logout":
    st.session_state.authenticated = False
    st.session_state.user_email = ""
    st.session_state.user_username = ""
    st.success("Session state arrays cleared.")
    st.rerun()

Overwriting app.py


In [10]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

Database initialized (tables ensured).
✅ Connected to PostgreSQL and ensured tables exist.


In [11]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
time.sleep(1)

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give the server a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("Open that URL in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://wick-flock-erased.ngrok-free.dev" -> "http://localhost:8501"
Open that URL in your browser. Leave this Colab cell/runtime running to keep it up.


In [ ]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
print("Stopped Streamlit and closed ngrok tunnel.")

Stopped Streamlit and closed ngrok tunnel.
